In [11]:
import torch
import torch.nn as nn   # ✅ THIS WAS MISSING
import timm

NUM_CLASSES = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
# Cell 8: Model Architecture
class CrossAttentionFusion(nn.Module):
    def __init__(self, feat_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn_L2R   = nn.MultiheadAttention(feat_dim, num_heads, dropout, batch_first=True)
        self.attn_R2L   = nn.MultiheadAttention(feat_dim, num_heads, dropout, batch_first=True)
        self.norm_L      = nn.LayerNorm(feat_dim)
        self.norm_R      = nn.LayerNorm(feat_dim)
        self.ffn_L       = nn.Sequential(nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(feat_dim*2, feat_dim),
                                         nn.Dropout(dropout))
        self.ffn_R       = nn.Sequential(nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(feat_dim*2, feat_dim),
                                         nn.Dropout(dropout))
        self.norm_ffn_L  = nn.LayerNorm(feat_dim)
        self.norm_ffn_R  = nn.LayerNorm(feat_dim)

    def forward(self, f_left, f_right):
        qL = f_left.unsqueeze(1);  qR = f_right.unsqueeze(1)
        aL, _ = self.attn_L2R(qL, qR, qR);  aL = aL.squeeze(1)
        fL = self.norm_L(f_left + aL)
        fL = self.norm_ffn_L(fL + self.ffn_L(fL))
        aR, _ = self.attn_R2L(qR, qL, qL);  aR = aR.squeeze(1)
        fR = self.norm_R(f_right + aR)
        fR = self.norm_ffn_R(fR + self.ffn_R(fR))
        return torch.cat([fL, fR], dim=1)

class MetadataMLP(nn.Module):
    def __init__(self, in_dim=2, hid=64, out=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid), nn.BatchNorm1d(hid), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(hid, out),   nn.BatchNorm1d(out),  nn.ReLU(True))
    def forward(self, x): return self.net(x)

class BilateralFusionNet(nn.Module):
    def __init__(self, num_classes=8, dropout=0.3):
        super().__init__()
        self.encoder = timm.create_model('efficientnet_b3', pretrained=True,
                                         num_classes=0, global_pool='avg')
        fd = self.encoder.num_features
        nh = 8 if fd % 8 == 0 else 4
        self.cross_attn  = CrossAttentionFusion(fd, nh, 0.1)
        self.meta_mlp    = MetadataMLP(2, 64, 128, 0.2)
        fusion_dim       = fd * 2 + 128
        self.classifier  = nn.Sequential(
            nn.Linear(fusion_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(512, 256),        nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(dropout/2),
            nn.Linear(256, num_classes))
        self.feat_dim    = fd
        self.num_classes = num_classes

    def forward(self, left, right, meta):
        fL = self.encoder(left)
        fR = self.encoder(right)
        fb = self.cross_attn(fL, fR)
        fm = self.meta_mlp(meta)
        return self.classifier(torch.cat([fb, fm], dim=1))

model = BilateralFusionNet(NUM_CLASSES).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'BilateralFusionNet | Parameters: {total:,}')
with torch.no_grad():
    o = model(torch.randn(2,3,224,224).to(DEVICE),
               torch.randn(2,3,224,224).to(DEVICE),
               torch.randn(2,2).to(DEVICE))
    print(f'Forward pass output shape: {o.shape}  (expected [2, 8])')

# ⚡ torch.compile — JIT-compiles the model graph (PyTorch 2.x)
# Provides 10-30% speedup by fusing operations and reducing overhead
import torch._dynamo
torch._dynamo.config.suppress_errors = True  # Graceful fallback if unsupported op
try:
    model_compiled = torch.compile(model, mode='reduce-overhead')
    # Quick warmup to trigger compilation
    with torch.no_grad(), torch.cuda.amp.autocast():
        _ = model_compiled(torch.randn(2,3,224,224).to(DEVICE),
                            torch.randn(2,3,224,224).to(DEVICE),
                            torch.randn(2,2).to(DEVICE))
    model = model_compiled
    print('✅ torch.compile() applied — graph-level optimization active.')
except Exception as e:
    print(f'⚠️  torch.compile() skipped ({e}) — using eager model.')

BilateralFusionNet | Parameters: 50,261,488
Forward pass output shape: torch.Size([2, 8])  (expected [2, 8])


/tmp/ipykernel_55/4247756833.py:76: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


✅ torch.compile() applied — graph-level optimization active.


In [20]:
import torch
import torch.nn as nn
import timm


MODEL_PATH = "/kaggle/input/models/lostvision/bilateralfusionnet-eye-disease-classifier-1/pytorch/default/1/best_model.pth"

model = BilateralFusionNet(num_classes=NUM_CLASSES).to(DEVICE)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
state_dict = checkpoint["model_state_dict"]

# 🔑 remove torch.compile prefix
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

model.load_state_dict(new_state_dict)

model.eval()

print("✅ Model loaded successfully")

✅ Model loaded successfully


In [48]:
import torch
import pandas as pd
import os
from PIL import Image
import torchvision.transforms as transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [49]:
df = pd.read_csv("/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k/full_df.csv")

In [50]:
labels = ['Normal','Diabetes','Glaucoma','Cataract',
          'AMD','Hypertension','Myopia','Other']

In [51]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [52]:
def test_one_patient(idx):
    row = df.iloc[idx]

    img_dir = "/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k/preprocessed_images"

    # Load images
    left_path  = os.path.join(img_dir, row['Left-Fundus'])
    right_path = os.path.join(img_dir, row['Right-Fundus'])

    left  = transform(Image.open(left_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    right = transform(Image.open(right_path).convert("RGB")).unsqueeze(0).to(DEVICE)

    # Metadata
    age = row['Patient Age'] / 100.0
    sex = 1 if row['Patient Sex'] == 'Male' else 0
    meta = torch.tensor([[age, sex]], dtype=torch.float32).to(DEVICE)

    # Prediction
    model.eval()
    with torch.no_grad():
        output = model(left, right, meta)
        probs = torch.sigmoid(output).cpu().numpy()[0]

    # Convert to disease names
    predicted = [labels[i] for i, p in enumerate(probs) if p > 0.5]

    # True labels
    true_labels = [
        labels[i]
        for i, col in enumerate(['N','D','G','C','A','H','M','O'])
        if row[col] == 1
    ]

    # Print result
    print(f"\n===== Patient {idx} =====")
    
    print("\nProbabilities:")
    for i, p in enumerate(probs):
        print(f"{labels[i]}: {p:.2f}")

    print("\nPredicted:", predicted)
    print("Actual   :", true_labels)

    if set(predicted) == set(true_labels):
        print("✅ Correct")
    else:
        print("❌ Incorrect")

In [ ]:
test_one_patient(1)


===== Patient 1 =====

Probabilities:
Normal: 0.65
Diabetes: 0.28
Glaucoma: 0.05
Cataract: 0.14
AMD: 0.26
Hypertension: 0.18
Myopia: 0.06
Other: 0.31

Predicted: ['Normal']
Actual   : ['Normal']
✅ Correct
